In order to run the bellow cells, download Amazon datasets for electronics from https://amazon-reviews-2023.github.io/main.html and place them in the /data folder.

In [ ]:
import json
import pandas as pd

In [ ]:
with open("../../data/Digital_Music.jsonl", "r") as f:
    fist_line = json.loads(f.readline())

In [ ]:
fist_line

### Filter Items that have been observed for the first time in year 2022 or later.

In [ ]:
def filter_data(data: dict) -> dict:
    filter = False

    if int(data["details"]["Date First Available"][-4:]) <2022:
        filter = True

    return filter

In [ ]:
with open("../../data/meta_Digital_Music.jsonl", "r") as fp:
    with open("../../data/meta_Digital_Music_2022_2023.jsonl", 'a', encoding='utf-8') as fp_out:
        with open("../../data/meta_Digital_Music_2022_2023_no_date.jsonl", 'a', encoding='utf-8') as fp_out_no_date:
            i = 0
            for line in fp:
                data = json.loads(line.strip())
                try:
                    filter = filter_data(data)
                    if not filter:
                        json.dump(data, fp_out)
                        fp_out.write('\n')
                        fp_out.flush()
                except:
                    json.dump(data, fp_out_no_date)
                    fp_out_no_date.write('\n')
                    fp_out_no_date.flush()

                i += 1
                if i % 10_000 == 0:
                    print(f"Processed {i} lines")

### Split the items into two categories: "has main category", "does not have main category"

In [ ]:
def filter_category(data: dict)-> dict :
    filter = False

    if (data["main_category"])== None:
        filter = True
        
    return filter

In [ ]:
with open("../../data/meta_Digital_Music.jsonl", 'r') as fp:
    with open("../../data/meta_Digital_Music_2022_2023_with_category.jsonl", 'a', encoding='utf-8') as fp_out:
        with open("../../data/meta_Digital_Music_2022_2023_no_category.jsonl", 'a', encoding='utf-8') as fp_out_no_category:
            for line in fp:
                data = json.loads(line.strip())
                if not filter_category(data):
                    json.dump(data, fp_out)
                    fp_out.write('\n')
                    fp_out.flush()
                else:
                    json.dump(data, fp_out_no_category)
                    fp_out_no_category.write('\n')
                    fp_out_no_category.flush()

### Explore distribution by categories

In [ ]:
df = pd.read_json("../../data/meta_Digital_Music_2022_2023_with_category.jsonl", lines=True)

In [ ]:
df["main_category"].value_counts().plot(kind="bar")

### Filter out items that have at least 100 ratings

In [ ]:
df_rating_100 = df[df["rating_number"] > 100]

In [ ]:
len(df)

In [ ]:
len(df_rating_100)

Explore distribution of ratings

In [ ]:
df_rating_100["average_rating"].plot(kind="hist", bins=50, range=(0, 5))

Sample 1000 items

In [ ]:
df_sample_1000 = df_rating_100.sample(n=1000, random_state=20)

In [ ]:
df_sample_1000["average_rating"].plot(kind="hist", bins=50, range=(0, 5))

In [ ]:
df_sample_1000["price"].plot(kind="hist", bins=100, range=(0, 500))

In [ ]:
df_sample_1000["main_category"].value_counts().plot(kind="bar")

In [ ]:
df_rating_100.to_json("../../data/meta_Digital_Music_2022_2023_with_category_rating_100.jsonl", orient="records", lines=True)

In [ ]:
df_rating_100.to_json("../../data/meta_Digital_Music_2022_2023_with_category_rating_100_sample_1000.jsonl", orient="records", lines=True)

Extract ratings that match sampled data.

In [ ]:
df_rating_100 = pd.read_json("../../data/meta_Digital_Music_2022_2023_with_category_rating_100.jsonl", lines=True)
df_rating_1000 = pd.read_json("../../data/meta_Digital_Music_2022_2023_with_category_rating_100_sample_1000.jsonl", lines=True)

In [ ]:
with open("../../data/Digital_Music.jsonl", 'r') as fp:
    with open("../../data/meta_Digital_Music_2022_2023_with_category_rating_100.jsonl", 'a') as fp_out:
        id_list = set(df_rating_100['parent_asin'].values)
        i = 0
        
        for line in fp:
            data = json.loads(line.strip())
            if data['parent_asin'] in id_list:
                json.dump(data, fp_out)
                fp_out.write('\n')
                fp_out.flush()
            i += 1
            if i % 100000 == 0:
                print(f"Processed {i} lines")

In [ ]:
with open("../../data/meta_Digital_Music_2022_2023_with_category_rating_100.jsonl", "r") as fp:
    with open("../../data/meta_Digital_Music_2022_2023_with_category_rating_100_sample_1000.jsonl", "a") as fp_out:
        id_list = set(df_sample_1000['parent_asin'].values)
        i = 0

        for line in fp:
            data = json.loads(line.strip())
            if data["parent_asin"] in id_list:
                json.dump(data, fp_out)
                fp_out.write("\n")
                fp_out.flush()

            i += 1
            if i % 100_000 == 0:
                print(f"Processed {i} lines")